# HAT K=6 — продолжение обучения на GPU
Выберите **Среда выполнения → Сменить среду выполнения → GPU**, затем запускайте ячейки сверху вниз.

Архив `hat-colab-transfer.zip` содержит код и локальную контрольную точку. Загружается один раз, затем остаётся в Google Drive. DeepGaze не изменяется. Веса и журналы сохраняются в `MyDrive/human-attention/hat-freeview-k6-seed42`. Данные загружаются из официального источника для исследовательского использования. Не публикуйте архив или данные.

Перенос сохраняет модель, AdamW, порядок данных и позицию шага. Смена PyTorch/устройства записывается в журнал; побитовое совпадение с Mac не обещается. Пакет остаётся 4.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, hashlib, zipfile, shutil, signal, time
import torch, torchvision
assert torch.cuda.is_available(), 'Выберите GPU в настройках среды выполнения и подключитесь заново.'
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)
drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/human-attention')
STORE.mkdir(parents=True, exist_ok=True)


In [ ]:
BUNDLE = STORE / 'hat-colab-transfer.zip'
EXPECTED = '258778b3bbc1787eb1d6fc04779cced53f105dd0b98195e3ee81fa9e0e447a7f'
if not BUNDLE.exists():
    from google.colab import files
    print('Выберите локальный файл hat-colab-transfer.zip')
    uploaded = files.upload()
    assert 'hat-colab-transfer.zip' in uploaded, 'Нужен файл hat-colab-transfer.zip'
    local = Path('/content/hat-colab-transfer.zip')
    assert hashlib.file_digest(local.open('rb'), 'sha256').hexdigest() == EXPECTED, 'Не тот архив'
    shutil.copyfile(local, BUNDLE)
    del uploaded
assert hashlib.file_digest(BUNDLE.open('rb'), 'sha256').hexdigest() == EXPECTED, 'Архив повреждён или другой версии'
WORK = Path('/content/human-attention')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    for name in z.namelist():
        dest = (WORK / name).resolve()
        assert dest.is_relative_to(WORK.resolve()), 'Небезопасный путь'
    z.extractall(WORK)
os.chdir(WORK)
if str(WORK) not in sys.path: sys.path.insert(0, str(WORK))
OUTPUT = STORE / 'hat-freeview-k6-seed42'
OUTPUT.mkdir(exist_ok=True)
if not (OUTPUT / 'latest.pt').exists():
    shutil.copyfile(WORK / 'checkpoint/latest.pt', OUTPUT / 'latest.pt')
state = torch.load(OUTPUT / 'latest.pt', map_location='cpu', weights_only=False)
print('Продолжаем с шага', state['step'], '| позиция в эпохе:', state['cursor'])
del state


In [ ]:
# Официальные архивы кэшируются в Drive; фотографии читаются с локального диска GPU-сервера.
from training import run_training
from training.prepare_images import prepare
DATA = WORK / 'data'
RAW = DATA / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
CACHE = STORE / 'dataset-cache'
CACHE.mkdir(exist_ok=True)
run_training.OUTPUT = OUTPUT
names = ['COCOFreeView_fixations_trainval.json', 'COCOSearch18-images-TP.zip', 'COCOSearch18-images-TA.zip']
for name in names:
    cached = CACHE / name
    run_training.download('http://vision.cs.stonybrook.edu/~cvlab_download/' + name, cached,
                          archive=name.endswith('.zip'),
                          hash_prefix='771167f4b5de5b9b58fb07c40d6e285d38b3b2ab9e4c6267df8a454a909da730' if name.endswith('.json') else None)
    if name.endswith('.json'):
        assert hashlib.file_digest(cached.open('rb'), 'sha256').hexdigest() == '771167f4b5de5b9b58fb07c40d6e285d38b3b2ab9e4c6267df8a454a909da730'
    shutil.copyfile(cached, RAW / name)
prepare(DATA)


In [ ]:
# Пробный шаг и небольшая валидация без изменения сохранённых весов.
ARGS = [sys.executable, '-u', '-m', 'training.train_hat', '--data', str(DATA),
        '--output', str(OUTPUT), '--device', 'cuda', '--batch', '4', '--epochs', '5',
        '--allow-transfer', '--save-seconds', '300']
subprocess.run(ARGS + ['--check-only'], check=True)


## Полное обучение
Журнал отображается ниже. После первых 25 шагов появится скорость и оценка времени на этой GPU. Сохранение — каждые 1000 шагов или 5 минут и при корректной остановке. Аварийное отключение Colab может потерять работу после последнего сохранения.

Чтобы остановить: нажмите квадрат остановки ячейки **один раз** и дождитесь сообщения о сохранении. Не удаляйте среду до завершения. Для продолжения в новом сеансе повторите ячейки сверху вниз; существующие веса в Drive не заменяются исходными.


In [ ]:
log = (OUTPUT / 'console-colab.log').open('a', buffering=1)
proc = subprocess.Popen(ARGS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, start_new_session=True)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    rc = proc.wait()
    if rc: raise RuntimeError(f'Обучение остановлено с кодом {rc}; проверьте вывод выше.')
except KeyboardInterrupt:
    print('Остановка: жду завершения шага и сохранения. Не отключайте среду…', flush=True)
    proc.send_signal(signal.SIGINT)
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    proc.wait()
finally:
    log.close()
print('Результаты:', OUTPUT)
